### 23. 合并K个升序链表
给你一个链表数组，每个链表都已经按升序排列。

请你将所有链表合并到一个升序链表中，返回合并后的链表。

示例 1：

输入：lists = [[1,4,5],[1,3,4],[2,6]]
输出：[1,1,2,3,4,4,5,6]

解释：链表数组如下：
[
  1->4->5,
  1->3->4,
  2->6
]

将它们合并到一个有序链表中得到。
1->1->2->3->4->4->5->6


#### 1. 最小堆
1. 合并后的第一个节点 first，一定是某个链表的头节点（因为链表已按升序排列）。
2. 合并后的第二个节点，可能是某个链表的头节点，也可能是 first 的下一个节点。
3. 按照这个过程继续思考，每当我们找到一个节点值最小的节点 x，就把节点 x.next 加入「可能是最小节点」的集合中。
4. 需要一个数据结构，支持：
   1.  从数据结构中找到并移除最小节点。
   2.  添加一个节点。
   
实现：**最小堆**
1. 初始把所有链表的头节点入堆；
2. 弹出最小节点，并把该节点的 next 节点入堆；
3. 重复步骤 2，直到堆为空。
4. 把弹出的节点按顺序拼接起来，就得到了答案。

In [ ]:
import heapq
from typing import List, Optional
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next
class Solution:
    def mergeKLists(self, lists: List[Optional[ListNode]]) -> Optional[ListNode]:
        heap = [] # 创建一个最小堆

        # 初始化堆，先加入各链表head
        for i, head in enumerate(lists): # 遍历lists中各链表
            if head: # 链表不为空时，将链表头节点加入堆
                heapq.heappush(heap, (head.val, i, head)) # 将节点值、链表索引和节点对象一起加入堆

        dummy = ListNode() # 创建一个虚拟头节点
        cur = dummy
        # 当堆不为空时，不断取出最小节点
        while heap:
            val, i, node = heapq.heappop(heap) # 取出堆顶元素，即当前最小节点
            # 链接最小链表
            cur.next = node
            cur = cur.next # 移动cur指针

            # 如果当前节点有下一个节点，将下一个节点加入堆
            if node.next: # 链表不为空时，将链表头节点加入堆
                heapq.heappush(heap, (node.next.val, i, node.next))
        
        return dummy.next
     

#### 2.分治 —— 递归
将各链表分组合并。
把 lists 一分为二（尽量均分），先合并前一半的链表，再合并后一半的链表，然后把这两个链表合并成最终的链表。

对 前半链表组 和 后半链表组 分别 递归继续分组 并合并。

In [ ]:
class Solution:
    # 1. 合并两个有序链表(双指针)
    def mergeTwoLists(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        dummy = ListNode() # 哑节点
        cur = dummy
        while l1 and l2: # 迭代
            if l1.val < l2.val: # 节点值比较
                cur.next = l1 # 创建节点
                l1 = l1.next # 迭代
            else:
                cur.next = l2
                l2 = l2.next
            cur = cur.next
        
        cur.next = l1 or l2
        return dummy.next

    # 递归调用分治 合并
    def mergeKLists(self, lists: List[Optional[ListNode]]) -> Optional[ListNode]:
        # 求链表组长度
        m = len(lists)
        if m == 0: # 链表为空时返回None
            return None

        if m == 1: # 链表组只有一个链表时直接返回该链表
            return lists[0]

        # 分治合并
        left = self.mergeKLists(lists[:m//2]) # 递归调用合并左半部分链表
        right = self.mergeKLists(lists[m//2:])
        return self.mergeTwoLists(left, right) # 合并左右两部分链表


#### 3. 分治——迭代
直接自底向上合并链表：

1. 两两合并：把 lists[0] 和 lists[1] 合并，合并后的链表保存在 lists[0] 中；把 lists[2] 和 lists[3] 合并，合并后的链表保存在 lists[2] 中；依此类推。
2. 四四合并：把 lists[0] 和 lists[2] 合并（相当于合并前四条链表），合并后的链表保存在 lists[0] 中；把 lists[4] 和 lists[6] 合并，合并后的链表保存在 lists[4] 中；依此类推。
3. 八八合并：把 lists[0] 和 lists[4] 合并（相当于合并前八条链表），合并后的链表保存在 lists[0] 中；把 lists[8] 和 lists[12] 合并，合并后的链表保存在 lists[8] 中；依此类推。
4. 依此类推，直到所有链表都合并到 lists[0] 中。最后返回 lists[0]。

作者：灵茶山艾府
链接：https://leetcode.cn/problems/merge-k-sorted-lists/solutions/2384305/liang-chong-fang-fa-zui-xiao-dui-fen-zhi-zbzx/

In [ ]:
class Solution:
    # 1. 合并两个有序链表(双指针)
    def mergeTwoLists(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        dummy = ListNode() # 创建一个虚拟头节点
        cur = dummy
        while l1 and l2: 
            if l1.val < l2.val: 
                cur.next = l1 
                l1 = l1.next 
            else: 
                cur.next = l2 
                l2 = l2.next 
            cur = cur.next 

        cur.next = l1 or l2
        return dummy.next
    
    # 2. 迭代调用分治 合并
    def mergeKLists(self, lists: List[Optional[ListNode]]) -> Optional[ListNode]:
        m = len(lists)
        if m == 0: # 链表组为空时返回None
            return None
        
        # 步长
        step = 1
        while step < m: # 链表长小于 m 时
            for i in range(0, m - step, step * 2): # 迭代合并链表
                lists[i] = self.mergeTwoLists(lists[i], lists[i + step]) # 合并链表
            step *= 2 # 步长翻倍
        return lists[0] # 返回合并后的链表
